<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-36.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 36 - Integración de datos + Python y R

**Módulo:** Sistemas de Big Data (SBD - 5074)

---

## Objetivos

- Integrar datos de múltiples fuentes heterogéneas
- Resolver conflictos de esquema y duplicados
- Comparar Python y R para análisis de datos
- Construir un pipeline ETL completo

---

## 1. Crear fuentes de datos simuladas

Vamos a simular 3 fuentes de datos con esquemas diferentes que debemos integrar.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import json
import hashlib
from datetime import datetime, timedelta

np.random.seed(42)

# === FUENTE 1: CSV de ventas (sistema de tienda online) ===
n_ventas = 500
fuente_csv = pd.DataFrame({
    'order_id': range(1000, 1000 + n_ventas),
    'customer_name': np.random.choice(
        ['Ana García', 'Luis Pérez', 'María López', 'Carlos Ruiz',
         'Elena Martín', 'Pedro Sánchez', 'Laura Torres', 'Juan Díaz',
         'Sofia Moreno', 'Diego Jiménez'], n_ventas),
    'customer_email': None,
    'product': np.random.choice(
        ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Auriculares'], n_ventas),
    'qty': np.random.randint(1, 5, n_ventas),
    'price': np.round(np.random.uniform(15, 500, n_ventas), 2),
    'date': [(datetime(2025, 6, 1) + timedelta(days=int(d))).strftime('%d/%m/%Y')
             for d in np.random.randint(0, 180, n_ventas)]
})

emails = {
    'Ana García': 'ana.garcia@mail.com', 'Luis Pérez': 'luis.perez@mail.com',
    'María López': 'maria.lopez@mail.com', 'Carlos Ruiz': 'carlos.ruiz@mail.com',
    'Elena Martín': 'elena.martin@mail.com', 'Pedro Sánchez': 'pedro.sanchez@mail.com',
    'Laura Torres': 'laura.torres@mail.com', 'Juan Díaz': 'juan.diaz@mail.com',
    'Sofia Moreno': 'sofia.moreno@mail.com', 'Diego Jiménez': 'diego.jimenez@mail.com'
}
fuente_csv['customer_email'] = fuente_csv['customer_name'].map(emails)

print("FUENTE 1 (CSV - Ventas online):")
print(f"  Columnas: {list(fuente_csv.columns)}")
print(f"  Registros: {len(fuente_csv)}")
print(fuente_csv.head(3))

# === FUENTE 2: JSON de productos (API de catálogo) ===
fuente_json = [
    {"id": "P001", "nombre": "Laptop", "categoria": "Electrónica",
     "precio_base": 450.00, "stock": 120, "proveedor": {"nombre": "TechCorp", "pais": "China"}},
    {"id": "P002", "nombre": "Mouse", "categoria": "Accesorios",
     "precio_base": 25.00, "stock": 500, "proveedor": {"nombre": "PerifCo", "pais": "Taiwan"}},
    {"id": "P003", "nombre": "Teclado", "categoria": "Accesorios",
     "precio_base": 45.00, "stock": 350, "proveedor": {"nombre": "PerifCo", "pais": "Taiwan"}},
    {"id": "P004", "nombre": "Monitor", "categoria": "Electrónica",
     "precio_base": 280.00, "stock": 80, "proveedor": {"nombre": "ScreenMax", "pais": "Corea"}},
    {"id": "P005", "nombre": "Auriculares", "categoria": "Accesorios",
     "precio_base": 60.00, "stock": 250, "proveedor": {"nombre": "AudioPro", "pais": "Japón"}}
]

print("\nFUENTE 2 (JSON - Catálogo):")
print(f"  Productos: {len(fuente_json)}")
print(f"  Ejemplo: {json.dumps(fuente_json[0], indent=2, ensure_ascii=False)}")

# === FUENTE 3: SQLite de clientes (CRM) ===
conn_crm = sqlite3.connect(':memory:')
conn_crm.execute('''
    CREATE TABLE clientes (
        id INTEGER PRIMARY KEY,
        nombre TEXT,
        email TEXT,
        ciudad TEXT,
        fecha_alta TEXT,
        segmento TEXT
    )
''')

clientes_crm = [
    (1, 'ANA GARCIA', 'ana.garcia@mail.com', 'Madrid', '2024-01-15', 'Premium'),
    (2, 'LUIS PEREZ', 'luis.perez@mail.com', 'Barcelona', '2024-03-20', 'Standard'),
    (3, 'MARIA LOPEZ', 'maria.lopez@mail.com', 'Sevilla', '2024-02-10', 'Premium'),
    (4, 'CARLOS RUIZ', 'carlos.ruiz@mail.com', 'Valencia', '2024-05-01', 'Standard'),
    (5, 'ELENA MARTIN', 'elena.martin@mail.com', 'Málaga', '2024-04-15', 'Premium'),
    (6, 'PEDRO SANCHEZ', 'pedro.sanchez@mail.com', 'Bilbao', '2024-06-01', 'Standard'),
    (7, 'LAURA TORRES', 'laura.torres@mail.com', 'Zaragoza', '2024-07-10', 'Standard'),
    (8, 'JUAN DIAZ', 'juan.diaz@mail.com', 'Madrid', '2024-01-20', 'Premium'),
    (9, 'SOFIA MORENO', 'sofia.moreno@mail.com', 'Sevilla', '2024-08-05', 'Standard'),
    (10, 'DIEGO JIMENEZ', 'diego.jimenez@mail.com', 'Barcelona', '2024-09-12', 'Premium'),
]
conn_crm.executemany('INSERT INTO clientes VALUES (?,?,?,?,?,?)', clientes_crm)
conn_crm.commit()

df_crm = pd.read_sql('SELECT * FROM clientes', conn_crm)
print("\nFUENTE 3 (SQLite - CRM):")
print(f"  Columnas: {list(df_crm.columns)}")
print(f"  Registros: {len(df_crm)}")
print(df_crm.head(3))

---

## 2. Estandarización de esquemas

Cada fuente usa nombres, formatos y convenciones diferentes. Debemos unificar.

In [ ]:
# === ESTANDARIZACIÓN ===

# Fuente 1 (CSV): normalizar nombres y formatos
df_ventas = fuente_csv.rename(columns={
    'order_id': 'pedido_id',
    'customer_name': 'nombre_cliente',
    'customer_email': 'email',
    'product': 'producto',
    'qty': 'cantidad',
    'price': 'precio',
    'date': 'fecha'
})
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'], format='%d/%m/%Y')
df_ventas['email'] = df_ventas['email'].str.lower().str.strip()

print("Ventas estandarizadas:")
print(df_ventas.dtypes)
print()

# Fuente 2 (JSON): aplanar estructura anidada
df_productos = pd.json_normalize(fuente_json, sep='_')
df_productos = df_productos.rename(columns={
    'id': 'producto_id',
    'nombre': 'producto',
    'precio_base': 'precio_catalogo',
    'proveedor_nombre': 'proveedor',
    'proveedor_pais': 'pais_proveedor'
})

print("Productos estandarizados:")
print(df_productos)
print()

# Fuente 3 (CRM): normalizar formato de nombres
df_clientes = df_crm.copy()
df_clientes['nombre'] = df_clientes['nombre'].str.title()
df_clientes['email'] = df_clientes['email'].str.lower().str.strip()
df_clientes['fecha_alta'] = pd.to_datetime(df_clientes['fecha_alta'])

print("Clientes estandarizados:")
print(df_clientes.head())

---

## 3. Integración: unir las 3 fuentes

Usamos el email como clave común para unir ventas con clientes, y el nombre de producto para unir con el catálogo.

In [ ]:
# JOIN 1: Ventas + Clientes (por email)
df_integrado = pd.merge(
    df_ventas,
    df_clientes[['email', 'ciudad', 'segmento']],
    on='email',
    how='left'
)

print("Tras JOIN ventas + clientes:")
print(f"  Registros: {len(df_integrado)}")
print(f"  Sin ciudad (no match): {df_integrado['ciudad'].isna().sum()}")

# JOIN 2: + Productos (por nombre producto)
df_integrado = pd.merge(
    df_integrado,
    df_productos[['producto', 'categoria', 'proveedor', 'pais_proveedor']],
    on='producto',
    how='left'
)

print(f"\nTras JOIN + productos:")
print(f"  Registros: {len(df_integrado)}")
print(f"  Columnas: {list(df_integrado.columns)}")

# Añadir campo calculado
df_integrado['importe_total'] = df_integrado['cantidad'] * df_integrado['precio']

print(f"\nDataset integrado final:")
print(df_integrado.head())
print(f"\n  Total registros: {len(df_integrado)}")
print(f"  Importe total: {df_integrado['importe_total'].sum():,.2f} €")

---

## 4. Deduplicación con Record Linkage

Detectar registros que podrían ser duplicados aunque tengan diferencias menores.

In [ ]:
# Simular datos con posibles duplicados
datos_con_dupes = pd.DataFrame({
    'nombre': ['Ana García', 'ANA GARCIA', 'Ana  García', 'Luis Pérez',
               'luis perez', 'María López', 'Maria Lopez', 'Carlos Ruiz'],
    'email': ['ana.garcia@mail.com', 'ana.garcia@mail.com', 'ana.garcia@gmail.com',
              'luis.perez@mail.com', 'luis.perez@mail.com', 'maria.lopez@mail.com',
              'maria.lopez@mail.com', 'carlos.ruiz@mail.com'],
    'fuente': ['CSV', 'CRM', 'Formulario', 'CSV', 'CRM', 'CSV', 'CRM', 'CSV']
})

print("Datos con posibles duplicados:")
print(datos_con_dupes)

def normalizar_nombre(nombre):
    """Normaliza nombre para comparación."""
    import unicodedata
    texto = nombre.lower().strip()
    texto = ' '.join(texto.split())
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    return texto

def generar_firma(row):
    """Genera hash para agrupar posibles duplicados."""
    nombre_norm = normalizar_nombre(row['nombre'])
    texto = f"{nombre_norm}|{row['email'].lower().strip()}"
    return hashlib.md5(texto.encode()).hexdigest()[:12]

datos_con_dupes['nombre_normalizado'] = datos_con_dupes['nombre'].apply(normalizar_nombre)
datos_con_dupes['firma'] = datos_con_dupes.apply(generar_firma, axis=1)

print("\nCon normalización y firma:")
print(datos_con_dupes[['nombre', 'nombre_normalizado', 'firma', 'fuente']])

# Deduplicar: quedarse con el primer registro de cada firma
dedup = datos_con_dupes.drop_duplicates(subset='firma', keep='first')
print(f"\nRegistros originales: {len(datos_con_dupes)}")
print(f"Registros únicos: {len(dedup)}")
print("\nResultado deduplicado:")
print(dedup[['nombre', 'email', 'fuente']])

---

## 5. Análisis con Python (Pandas + Matplotlib)

Análisis completo sobre el dataset integrado usando el stack Python.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Ventas por categoría
ventas_cat = df_integrado.groupby('categoria')['importe_total'].sum().sort_values()
ventas_cat.plot(kind='barh', ax=axes[0, 0], color=['#2ecc71', '#3498db'])
axes[0, 0].set_title('Ventas por categoría')
axes[0, 0].set_xlabel('Importe total (€)')

# 2. Ventas por segmento de cliente
ventas_seg = df_integrado.groupby('segmento')['importe_total'].sum()
ventas_seg.plot(kind='pie', ax=axes[0, 1], autopct='%1.1f%%',
                colors=['#e74c3c', '#3498db'])
axes[0, 1].set_title('Ventas por segmento')
axes[0, 1].set_ylabel('')

# 3. Evolución temporal
ventas_diarias = df_integrado.set_index('fecha').resample('W')['importe_total'].sum()
ventas_diarias.plot(ax=axes[1, 0], color='#9b59b6', linewidth=2)
axes[1, 0].set_title('Ventas semanales')
axes[1, 0].set_ylabel('Importe (€)')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Distribución de importes por producto
df_integrado.boxplot(column='importe_total', by='producto', ax=axes[1, 1])
axes[1, 1].set_title('Distribución importes por producto')
axes[1, 1].set_xlabel('Producto')
axes[1, 1].set_ylabel('Importe (€)')
plt.suptitle('')

plt.tight_layout()
plt.show()

# Resumen estadístico
print("\n=== RESUMEN ESTADÍSTICO ===")
resumen = df_integrado.groupby('producto').agg(
    pedidos=('pedido_id', 'count'),
    importe_total=('importe_total', 'sum'),
    ticket_medio=('importe_total', 'mean'),
    cantidad_total=('cantidad', 'sum')
).round(2).sort_values('importe_total', ascending=False)

print(resumen)

---

## 6. Equivalencias Python ↔ R

El mismo análisis anterior expresado en R (como referencia).

| Operación | Python (Pandas) | R (Tidyverse) |
|-----------|----------------|----------------|
| Leer CSV | `pd.read_csv('f.csv')` | `read_csv('f.csv')` |
| Filtrar | `df[df['col'] > 5]` | `df %>% filter(col > 5)` |
| Agrupar + agregar | `df.groupby('c').agg(...)` | `df %>% group_by(c) %>% summarise(...)` |
| Ordenar | `df.sort_values('c')` | `df %>% arrange(c)` |
| Unir tablas | `pd.merge(a, b, on='k')` | `left_join(a, b, by='k')` |
| Gráfico barras | `df.plot(kind='bar')` | `ggplot(...) + geom_col()` |
| Modelo lineal | `LinearRegression().fit(X,y)` | `lm(y ~ x, data=df)` |

```r
# Equivalente R del análisis anterior
library(tidyverse)

# Cargar y limpiar
ventas <- read_csv('ventas.csv') %>%
  mutate(fecha = dmy(date),
         email = str_to_lower(str_trim(customer_email)))

# Integrar
integrado <- ventas %>%
  left_join(clientes, by = 'email') %>%
  left_join(productos, by = 'producto') %>%
  mutate(importe_total = qty * price)

# Resumen
integrado %>%
  group_by(producto) %>%
  summarise(
    pedidos = n(),
    importe_total = sum(importe_total),
    ticket_medio = mean(importe_total)
  ) %>%
  arrange(desc(importe_total))

# Gráfico
ggplot(integrado, aes(x = producto, y = importe_total, fill = categoria)) +
  geom_boxplot() +
  labs(title = 'Distribución importes por producto') +
  theme_minimal()
```

---

## 7. Pipeline ETL completo: de fuentes a base de datos analítica

In [ ]:
def pipeline_etl(df_ventas, df_productos, df_clientes):
    """
    Pipeline ETL completo:
    Extract → Transform → Load
    """
    print("="*50)
    print("PIPELINE ETL")
    print("="*50)

    # --- EXTRACT ---
    print("\n📥 EXTRACT")
    print(f"  Ventas: {len(df_ventas)} registros")
    print(f"  Productos: {len(df_productos)} registros")
    print(f"  Clientes: {len(df_clientes)} registros")

    # --- TRANSFORM ---
    print("\n🔄 TRANSFORM")

    # Validar: eliminar registros con campos críticos nulos
    antes = len(df_ventas)
    df_v = df_ventas.dropna(subset=['email', 'producto', 'cantidad', 'precio'])
    print(f"  Validación: {antes - len(df_v)} registros eliminados por nulos")

    # Integrar fuentes
    df_merged = pd.merge(df_v, df_clientes[['email', 'ciudad', 'segmento']],
                         on='email', how='left')
    df_merged = pd.merge(df_merged, df_productos[['producto', 'categoria', 'proveedor']],
                         on='producto', how='left')

    # Enriquecer
    df_merged['importe'] = df_merged['cantidad'] * df_merged['precio']
    df_merged['mes'] = df_merged['fecha'].dt.to_period('M').astype(str)
    df_merged['dia_semana'] = df_merged['fecha'].dt.day_name()

    print(f"  Integración: {len(df_merged)} registros resultantes")
    print(f"  Columnas finales: {len(df_merged.columns)}")

    # --- LOAD ---
    print("\n📤 LOAD")
    conn_dw = sqlite3.connect(':memory:')

    # Tabla de hechos
    df_merged.to_sql('fact_ventas', conn_dw, if_exists='replace', index=False)

    # Tabla resumen mensual
    resumen = df_merged.groupby(['mes', 'producto', 'categoria', 'segmento']).agg(
        num_pedidos=('pedido_id', 'count'),
        importe_total=('importe', 'sum'),
        cantidad_total=('cantidad', 'sum')
    ).reset_index()
    resumen.to_sql('dim_resumen_mensual', conn_dw, if_exists='replace', index=False)

    print(f"  fact_ventas: {len(df_merged)} registros cargados")
    print(f"  dim_resumen_mensual: {len(resumen)} registros cargados")

    # Verificación
    print("\n✅ VERIFICACIÓN")
    check = pd.read_sql('SELECT COUNT(*) as n FROM fact_ventas', conn_dw)
    print(f"  Registros en DWH: {check['n'].iloc[0]}")

    top_query = '''
        SELECT producto, segmento,
               COUNT(*) as pedidos,
               ROUND(SUM(importe), 2) as total
        FROM fact_ventas
        GROUP BY producto, segmento
        ORDER BY total DESC
        LIMIT 10
    '''
    print("\n  Top 10 (producto × segmento):")
    print(pd.read_sql(top_query, conn_dw).to_string(index=False))

    conn_dw.close()
    print("\n" + "="*50)
    print("✅ Pipeline ETL completado")
    print("="*50)

pipeline_etl(df_ventas, df_productos, df_clientes)

---

## Recursos

- [Pandas - Merging DataFrames](https://pandas.pydata.org/docs/user_guide/merging.html)
- [R for Data Science](https://r4ds.had.co.nz/)
- [Tidyverse](https://www.tidyverse.org/)
- [Data Integration Patterns](https://martinfowler.com/articles/data-monolith-to-mesh.html)